# Cycle & Sleep: Hormonal Patterns in Sleep Quality

Investigamos si el ciclo menstrual influye en la duración, estructura y calidad del sueño.

**Datos**: Apple Health export (~6 años)  
**Métricas de sueño**: duración total, % REM, % Deep, eficiencia, despertares  
**Métricas del ciclo**: fase (menstrual, folicular, ovulatoria, lútea), temperatura de muñeca

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import timedelta

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

DATA_DIR = "../data"

## 1. Cargar datos

In [ ]:
sleep_raw = pd.read_csv(f"{DATA_DIR}/sleep.csv", parse_dates=["start", "end"])
menstrual = pd.read_csv(f"{DATA_DIR}/menstrual.csv", parse_dates=["date"])
wrist_temp = pd.read_csv(f"{DATA_DIR}/wrist_temp.csv", parse_dates=["date"])
breathing = pd.read_csv(f"{DATA_DIR}/breathing.csv", parse_dates=["date"])

print(f"Sleep records: {len(sleep_raw):,}")
print(f"Menstrual records: {len(menstrual):,}")
print(f"Wrist temp records: {len(wrist_temp):,}")
print(f"Breathing records: {len(breathing):,}")
print(f"\nSleep stages: {sleep_raw['stage'].value_counts().to_dict()}")
print(f"Flow types: {menstrual['flow'].value_counts().to_dict()}")
print(f"\nSleep date range: {sleep_raw['start'].min().date()} → {sleep_raw['start'].max().date()}")
print(f"Menstrual date range: {menstrual['date'].min().date()} → {menstrual['date'].max().date()}")

## 2. Agregar sueño por noche

Cada noche se asigna a la fecha en que empieza el sueño (si empiezas a dormir a las 23:00 del día 5, esa noche cuenta como día 5). Para sueño que empieza después de medianoche pero antes de las 6:00, se asigna al día anterior.

In [ ]:
# Assign each sleep record to a "sleep night" date
sleep = sleep_raw.copy()
sleep["hour"] = sleep["start"].dt.hour
# If sleep starts between midnight and 6am, assign to previous day
sleep["night_date"] = sleep["start"].dt.date
mask_early = sleep["hour"] < 6
sleep.loc[mask_early, "night_date"] = (sleep.loc[mask_early, "start"] - timedelta(days=1)).dt.date
sleep["night_date"] = pd.to_datetime(sleep["night_date"])

# Aggregate per night
stages_of_interest = ["AsleepCore", "AsleepREM", "AsleepDeep", "Awake", "InBed"]

nightly = []
for night, group in sleep.groupby("night_date"):
    row = {"night_date": night}
    
    for stage in stages_of_interest:
        stage_data = group[group["stage"] == stage]
        row[f"{stage}_min"] = stage_data["duration_min"].sum()
    
    # Total sleep = Core + REM + Deep (+ Unspecified if present)
    unspecified = group[group["stage"] == "AsleepUnspecified"]["duration_min"].sum()
    row["total_sleep_min"] = row["AsleepCore_min"] + row["AsleepREM_min"] + row["AsleepDeep_min"] + unspecified
    
    # Total in-bed: use InBed if available, otherwise sum all stages
    total_all = row["total_sleep_min"] + row["Awake_min"]
    row["total_inbed_min"] = row["InBed_min"] if row["InBed_min"] >= total_all else total_all
    
    # Percentages
    if row["total_sleep_min"] > 0:
        row["pct_rem"] = row["AsleepREM_min"] / row["total_sleep_min"] * 100
        row["pct_deep"] = row["AsleepDeep_min"] / row["total_sleep_min"] * 100
        row["pct_core"] = row["AsleepCore_min"] / row["total_sleep_min"] * 100
    else:
        row["pct_rem"] = row["pct_deep"] = row["pct_core"] = np.nan
    
    # Efficiency: sleep time / in-bed time (cap at 100%)
    if row["total_inbed_min"] > 0:
        row["efficiency"] = min(row["total_sleep_min"] / row["total_inbed_min"] * 100, 100.0)
    else:
        row["efficiency"] = np.nan
    
    # Number of awakenings
    row["n_awakenings"] = len(group[group["stage"] == "Awake"])
    
    # Sleep start/end times
    row["sleep_start"] = group["start"].min()
    row["sleep_end"] = group["end"].max()
    
    nightly.append(row)

nightly_df = pd.DataFrame(nightly)

# Filter: only nights with meaningful sleep data (>2h sleep, <16h in bed)
nightly_df = nightly_df[
    (nightly_df["total_sleep_min"] > 120) &
    (nightly_df["total_inbed_min"] < 960)
].reset_index(drop=True)

print(f"Noches válidas: {len(nightly_df)}")
print(f"Eficiencia media: {nightly_df['efficiency'].mean():.1f}% (rango: {nightly_df['efficiency'].min():.1f}% - {nightly_df['efficiency'].max():.1f}%)")
nightly_df.head()

## 3. Identificar ciclos menstruales y asignar fases

Agrupamos los días de sangrado consecutivos en periodos, y estimamos las fases del ciclo entre cada periodo.

In [ ]:
# Identify period start dates (first day of bleeding after a gap of >5 days)
menstrual_sorted = menstrual.sort_values("date").reset_index(drop=True)
menstrual_sorted["date_dt"] = pd.to_datetime(menstrual_sorted["date"])
menstrual_sorted["gap"] = menstrual_sorted["date_dt"].diff().dt.days

# A new period starts when gap > 5 days (or first record)
menstrual_sorted["new_period"] = (menstrual_sorted["gap"] > 5) | (menstrual_sorted["gap"].isna())
menstrual_sorted["period_id"] = menstrual_sorted["new_period"].cumsum()

# Get period start dates and durations
periods = menstrual_sorted.groupby("period_id").agg(
    start=('date_dt', 'min'),
    end=('date_dt', 'max'),
    n_days=('date_dt', 'count')
).reset_index()

# Cycle length = days between period starts
periods["cycle_length"] = periods["start"].diff().dt.days

print(f"Periodos identificados: {len(periods)}")
print(f"Duración media del ciclo: {periods['cycle_length'].mean():.1f} días (σ={periods['cycle_length'].std():.1f})")
print(f"Duración media del sangrado: {periods['n_days'].mean():.1f} días")
print()
periods.tail(10)

In [ ]:
# Visualize cycle length over time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
valid_cycles = periods.dropna(subset=["cycle_length"])
ax.plot(valid_cycles["start"], valid_cycles["cycle_length"], "o-", markersize=4, alpha=0.7)
ax.axhline(y=28, color="red", linestyle="--", alpha=0.5, label="28 días")
ax.set_title("Duración del ciclo a lo largo del tiempo")
ax.set_ylabel("Días")
ax.legend()

ax = axes[1]
ax.hist(valid_cycles["cycle_length"], bins=20, edgecolor="white", alpha=0.8)
ax.axvline(x=valid_cycles["cycle_length"].median(), color="red", linestyle="--", label=f"Mediana: {valid_cycles['cycle_length'].median():.0f}d")
ax.set_title("Distribución de la duración del ciclo")
ax.set_xlabel("Días")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
def assign_cycle_phase(date, periods_df):
    """
    Assign a cycle phase to a given date based on period data.
    
    Phases (proportional to cycle length):
    - Menstrual: days 1 to bleeding_end
    - Follicular: bleeding_end+1 to ~day 13 (proportional)
    - Ovulatory: ~days 14-16 (proportional)
    - Luteal: ~day 17 to cycle end (proportional)
    """
    date = pd.Timestamp(date)
    
    for i in range(len(periods_df) - 1):
        period_start = periods_df.iloc[i]["start"]
        period_end = periods_df.iloc[i]["end"]
        next_period_start = periods_df.iloc[i + 1]["start"]
        cycle_length = (next_period_start - period_start).days
        
        if period_start <= date < next_period_start:
            day_in_cycle = (date - period_start).days + 1
            bleeding_days = (period_end - period_start).days + 1
            
            if day_in_cycle <= bleeding_days:
                return "Menstrual", day_in_cycle, cycle_length, i
            
            # Proportional phase boundaries
            follicular_end = int(cycle_length * 0.46)   # ~day 13 of 28
            ovulatory_end = int(cycle_length * 0.57)    # ~day 16 of 28
            
            if day_in_cycle <= follicular_end:
                return "Folicular", day_in_cycle, cycle_length, i
            elif day_in_cycle <= ovulatory_end:
                return "Ovulatoria", day_in_cycle, cycle_length, i
            else:
                return "Lútea", day_in_cycle, cycle_length, i
    
    return None, None, None, None


# Assign phases to each night
phases = nightly_df["night_date"].apply(lambda d: assign_cycle_phase(d, periods))
nightly_df["phase"] = phases.apply(lambda x: x[0])
nightly_df["cycle_day"] = phases.apply(lambda x: x[1])
nightly_df["cycle_length"] = phases.apply(lambda x: x[2])
nightly_df["cycle_id"] = phases.apply(lambda x: x[3])

# Filter to only nights within a known cycle
cycle_sleep = nightly_df.dropna(subset=["phase"]).copy()
# Filter out abnormal cycles (<21 or >45 days)
cycle_sleep = cycle_sleep[(cycle_sleep["cycle_length"] >= 21) & (cycle_sleep["cycle_length"] <= 45)]

print(f"Noches con fase del ciclo asignada: {len(cycle_sleep)}")
print(f"\nDistribución por fase:")
print(cycle_sleep["phase"].value_counts())

## 4. Merge con temperatura y respiración

In [ ]:
# Merge wrist temperature
wrist_temp["night_date"] = pd.to_datetime(wrist_temp["date"])
cycle_sleep = cycle_sleep.merge(wrist_temp[["night_date", "temp_c"]], on="night_date", how="left")

# Merge breathing disturbances
breathing["night_date"] = pd.to_datetime(breathing["date"])
cycle_sleep = cycle_sleep.merge(breathing[["night_date", "disturbances"]], on="night_date", how="left")

print(f"Noches con temperatura: {cycle_sleep['temp_c'].notna().sum()}")
print(f"Noches con respiración: {cycle_sleep['disturbances'].notna().sum()}")

## 5. Análisis descriptivo por fase del ciclo

In [ ]:
phase_order = ["Menstrual", "Folicular", "Ovulatoria", "Lútea"]
metrics = ["total_sleep_min", "pct_rem", "pct_deep", "pct_core", "efficiency", "n_awakenings"]
metric_labels = {
    "total_sleep_min": "Duración total (min)",
    "pct_rem": "% REM",
    "pct_deep": "% Deep",
    "pct_core": "% Core",
    "efficiency": "Eficiencia (%)",
    "n_awakenings": "Despertares"
}

summary = cycle_sleep.groupby("phase")[metrics].agg(["mean", "std", "median", "count"]).round(2)
summary = summary.reindex(phase_order)
summary

In [ ]:
# Box plots for each metric by cycle phase
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = {"Menstrual": "#e74c3c", "Folicular": "#3498db", "Ovulatoria": "#2ecc71", "Lútea": "#f39c12"}

for ax, metric in zip(axes.flat, metrics):
    data = [cycle_sleep[cycle_sleep["phase"] == p][metric].dropna() for p in phase_order]
    bp = ax.boxplot(data, tick_labels=phase_order, patch_artist=True, widths=0.6)
    for patch, phase in zip(bp["boxes"], phase_order):
        patch.set_facecolor(colors[phase])
        patch.set_alpha(0.6)
    ax.set_title(metric_labels[metric], fontsize=13, fontweight="bold")
    ax.tick_params(axis="x", rotation=15)

fig.suptitle("Métricas de sueño por fase del ciclo menstrual", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 6. Tests estadísticos

Kruskal-Wallis test (no-paramétrico) para cada métrica entre las 4 fases, seguido de tests post-hoc de Dunn si hay significancia.

In [ ]:
from itertools import combinations

print("=" * 70)
print("KRUSKAL-WALLIS TEST: Diferencias entre fases del ciclo")
print("=" * 70)

significant_metrics = []

for metric in metrics:
    groups = [cycle_sleep[cycle_sleep["phase"] == p][metric].dropna() for p in phase_order]
    stat, p_val = stats.kruskal(*groups)
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
    print(f"\n{metric_labels[metric]}:")
    print(f"  H = {stat:.3f}, p = {p_val:.6f} {sig}")
    
    if p_val < 0.05:
        significant_metrics.append(metric)
        # Post-hoc: pairwise Mann-Whitney U tests with Bonferroni correction
        n_comparisons = 6  # C(4,2)
        for (i, p1), (j, p2) in combinations(enumerate(phase_order), 2):
            u_stat, u_p = stats.mannwhitneyu(groups[i], groups[j], alternative="two-sided")
            adj_p = min(u_p * n_comparisons, 1.0)  # Bonferroni
            if adj_p < 0.05:
                print(f"    {p1} vs {p2}: U={u_stat:.0f}, p_adj={adj_p:.4f} {'*' if adj_p < 0.05 else ''}")

print(f"\n\nMétricas con diferencias significativas: {len(significant_metrics)}")

## 7. Temperatura de muñeca por fase del ciclo

La temperatura basal sube en la fase lútea por la progesterona. Veamos si esto se refleja en la temperatura de muñeca durante el sueño.

In [ ]:
temp_data = cycle_sleep.dropna(subset=["temp_c"])

if len(temp_data) > 20:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Box plot by phase
    ax = axes[0]
    data = [temp_data[temp_data["phase"] == p]["temp_c"].dropna() for p in phase_order]
    bp = ax.boxplot(data, tick_labels=phase_order, patch_artist=True, widths=0.6)
    for patch, phase in zip(bp["boxes"], phase_order):
        patch.set_facecolor(colors[phase])
        patch.set_alpha(0.6)
    ax.set_title("Temperatura de muñeca por fase", fontsize=13, fontweight="bold")
    ax.set_ylabel("°C")
    
    # Mean temp over cycle day
    ax = axes[1]
    # Normalize cycle day to 0-1 scale
    temp_data = temp_data.copy()
    temp_data["cycle_day_norm"] = temp_data["cycle_day"] / temp_data["cycle_length"]
    temp_data_sorted = temp_data.sort_values("cycle_day_norm")
    # Rolling average
    ax.scatter(temp_data_sorted["cycle_day_norm"], temp_data_sorted["temp_c"], alpha=0.3, s=20)
    # Bin and average
    temp_data["bin"] = pd.cut(temp_data["cycle_day_norm"], bins=20)
    binned = temp_data.groupby("bin", observed=True)["temp_c"].mean()
    bin_centers = [interval.mid for interval in binned.index]
    ax.plot(bin_centers, binned.values, "r-o", linewidth=2, markersize=5, label="Media por bin")
    ax.axvspan(0, 0.18, alpha=0.1, color="red", label="Menstrual")
    ax.axvspan(0.46, 0.57, alpha=0.1, color="green", label="Ovulatoria")
    ax.set_title("Temperatura a lo largo del ciclo (normalizado)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Posición en el ciclo (0=inicio, 1=fin)")
    ax.set_ylabel("°C")
    ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Statistical test
    groups = [temp_data[temp_data["phase"] == p]["temp_c"].dropna() for p in phase_order]
    stat, p_val = stats.kruskal(*[g for g in groups if len(g) > 0])
    print(f"Kruskal-Wallis temperatura por fase: H={stat:.3f}, p={p_val:.6f}")
else:
    print(f"Insuficientes datos de temperatura con fase asignada ({len(temp_data)} registros)")

## 8. Evolución del sueño a lo largo del ciclo

Visualizamos cómo cambian las métricas de sueño a medida que avanza el ciclo (día 1 → último día).

In [ ]:
# Normalize cycle day position (0 to 1)
cycle_sleep_norm = cycle_sleep.copy()
cycle_sleep_norm["cycle_pos"] = cycle_sleep_norm["cycle_day"] / cycle_sleep_norm["cycle_length"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_metrics = ["total_sleep_min", "pct_rem", "pct_deep", "efficiency"]

for ax, metric in zip(axes.flat, plot_metrics):
    # Bin into 20 equal segments of the cycle
    temp = cycle_sleep_norm[["cycle_pos", metric]].dropna()
    temp["bin"] = pd.cut(temp["cycle_pos"], bins=20)
    binned = temp.groupby("bin", observed=True)[metric].agg(["mean", "sem"])
    bin_centers = [interval.mid for interval in binned.index]
    
    ax.fill_between(bin_centers, 
                    binned["mean"] - 1.96 * binned["sem"],
                    binned["mean"] + 1.96 * binned["sem"],
                    alpha=0.2, color="steelblue")
    ax.plot(bin_centers, binned["mean"], "o-", color="steelblue", linewidth=2, markersize=4)
    
    # Phase boundaries
    ax.axvspan(0, 0.18, alpha=0.08, color="red")
    ax.axvspan(0.46, 0.57, alpha=0.08, color="green")
    ax.axvspan(0.57, 1.0, alpha=0.08, color="orange")
    
    ax.set_title(metric_labels[metric], fontsize=13, fontweight="bold")
    ax.set_xlabel("Posición en el ciclo")

fig.suptitle("Evolución del sueño a lo largo del ciclo menstrual", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 9. Correlación entre métricas

In [ ]:
corr_cols = ["total_sleep_min", "pct_rem", "pct_deep", "pct_core", "efficiency", 
             "n_awakenings", "temp_c", "disturbances", "cycle_day"]
corr_labels = {
    "total_sleep_min": "Duración",
    "pct_rem": "% REM",
    "pct_deep": "% Deep",
    "pct_core": "% Core",
    "efficiency": "Eficiencia",
    "n_awakenings": "Despertares",
    "temp_c": "Temp. muñeca",
    "disturbances": "Pert. respiratorias",
    "cycle_day": "Día del ciclo"
}

corr_data = cycle_sleep[corr_cols].dropna()
corr_matrix = corr_data.corr(method="spearman")
corr_matrix.index = [corr_labels.get(c, c) for c in corr_matrix.index]
corr_matrix.columns = [corr_labels.get(c, c) for c in corr_matrix.columns]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlaciones de Spearman entre métricas de sueño y ciclo", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 10. Efecto premenstrual: Late Luteal vs Early Luteal

La fase lútea tardía (últimos 5 días antes del periodo) es cuando los síntomas premenstruales son más intensos. Separamos la fase lútea en dos para ver si el sueño empeora justo antes del periodo.

In [ ]:
# Split luteal phase into early vs late (last 5 days before next period)
luteal = cycle_sleep[cycle_sleep["phase"] == "Lútea"].copy()
luteal["days_to_next_period"] = luteal["cycle_length"] - luteal["cycle_day"]
luteal["luteal_sub"] = np.where(luteal["days_to_next_period"] <= 5, "Premenstrual\n(<=5d)", "Lútea temprana\n(>5d)")

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
premenstrual_metrics = ["total_sleep_min", "pct_rem", "n_awakenings", "efficiency"]
premenstrual_labels = ["Duración (min)", "% REM", "Despertares", "Eficiencia (%)"]
sub_colors = {"Premenstrual\n(<=5d)": "#e74c3c", "Lútea temprana\n(>5d)": "#f39c12"}
sub_order = ["Lútea temprana\n(>5d)", "Premenstrual\n(<=5d)"]

for ax, metric, label in zip(axes, premenstrual_metrics, premenstrual_labels):
    data = [luteal[luteal["luteal_sub"] == s][metric].dropna() for s in sub_order]
    bp = ax.boxplot(data, tick_labels=sub_order, patch_artist=True, widths=0.5)
    for patch, s in zip(bp["boxes"], sub_order):
        patch.set_facecolor(sub_colors[s])
        patch.set_alpha(0.6)
    ax.set_title(label, fontsize=13, fontweight="bold")
    
    # Mann-Whitney test
    u, p = stats.mannwhitneyu(data[0], data[1], alternative="two-sided")
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax.text(0.5, 0.02, f"p={p:.4f} {sig}", transform=ax.transAxes, ha="center", fontsize=10, 
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray"))

fig.suptitle("Efecto premenstrual: últimos 5 días antes del periodo vs resto de fase lútea",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Print summary
print(f"Noches lútea temprana: {len(luteal[luteal['luteal_sub'] == 'Lútea temprana\\n(>5d)'])}")
print(f"Noches premenstruales: {len(luteal[luteal['luteal_sub'] == 'Premenstrual\\n(<=5d)'])}")
for metric, label in zip(premenstrual_metrics, premenstrual_labels):
    early = luteal[luteal["days_to_next_period"] > 5][metric].dropna()
    late = luteal[luteal["days_to_next_period"] <= 5][metric].dropna()
    u, p = stats.mannwhitneyu(early, late, alternative="two-sided")
    print(f"  {label}: early={early.mean():.1f} vs pre={late.mean():.1f} (p={p:.4f})")

## 11. Resumen de hallazgos